# SupplementRx AI — Full KG Construction

**Status: Steps 1–2 completed; Step 3 rule-based mapping inventory completed with unresolved cases retained. Step 4 source-preserving identity/duplicate review completed; unsupported equivalence remains unresolved. Step 5 expanded graph exported and integrity-checked. Step 6 representative retrieval checks completed; a limited POC baseline is frozen with documented gaps. See Step 6 for unresolved issues. See the stage checkpoints for current scope and limits.**

“Full” means eligible records from the current ten-supplement collection, including co-ingredients and source-qualified content. It does not mean every raw record must become an active graph assertion. Uncertain mappings remain pending/excluded, and clinical annotations retain their review status.

This notebook organizes the remaining preparation and construction work. Reusable scripts will perform transformations; this notebook will run, inspect and explain them. The existing [pilot notebook](Preprocessing_Pilot.ipynb) preserves the experiments and fixed answer examples.

## Workflow

1. **Preparation coverage:** identify completed, pilot-only and pending work for all six sources.
2. **Expand normalization:** apply selected-field, identifier and supported-unit rules in batches.
3. **Complete ingredient mapping:** review cross-source candidates with relation-specific High/Medium/Low confidence.
4. **Resolve identities and duplicates:** distinguish replays, labels/versions, formulations and repeated evidence; merge only with explicit support.
5. **Build the expanded graph:** export eligible nodes, relationships, provenance and review queues.
6. **Validate and freeze a POC version:** verify integrity, coverage and representative questions; record unresolved issues.

Embeddings, GCN/R-GCN, a general answering layer and database import are later decisions. No such work is started here.

### Working rules

- Preserve raw data and previous builds; pin input hashes and rule versions.
- Retain original values, unknown fields and source conflicts alongside derived representations.
- Concept, exact-material, topic and related-subject links retain different meanings.
- High technical confidence is not clinical approval. Pending clinical annotations remain attributed source content.
- Reddit mentions/context are not evidence of personal use, effectiveness or safety.

**Navigation revision:** explicit DSLD title/name mentions now cover all ten topics; Probiotics has 40 reachable labels. See the final section for the new graph and release. Earlier stage results describe the preserved baseline.

**Latest revision:** collection-wide statement preservation fixes the demonstrated warning export gap. See the final section for the current graph and evaluation bindings; earlier outputs describe preserved historical versions.


## Step 1 — Preparation coverage and evidence

**Baseline audits and provisional POC field selections exist for all six datasets.** The shared design and bounded preprocessing are implemented, but collection-wide normalization, identity decisions and graph export are unfinished.

Status meanings:

- **Complete for saved snapshot:** the stated audit/inventory task covers its saved collection; it does not validate clinical truth or present-day source accuracy.
- **Pilot-only:** implemented and checked on selected records, not the full collection.
- **Pending:** not yet applied across the selected POC fields/records.

References: [audit/field selections](Data_Audit_and_Preprocessing.ipynb), [schema/rules](Schema_Design_and_Mapping.ipynb), [pilot implementation](Preprocessing_Pilot.ipynb), [bounded contract](../schemas/poc_preprocessing_contract_v0.1.json).

The counts below describe graph representation, not quality, recall or the number of records that must ultimately be exported. MeSH counts refer to matched records, not every RDF resource. Clinical claims and monographs have separate denominators.


In [1]:
from pathlib import Path
from collections import Counter
import json, csv, hashlib
import pandas as pd

kg_root = Path.cwd()
if not (kg_root / 'data').is_dir():
    kg_root = kg_root.parent
csv.field_size_limit(10_000_000)
kg_build = kg_root / 'data/graph/creatine-natmed-v1-f174e5e28b21603a'
kg_manifest = json.loads((kg_build / 'manifest.json').read_text())
for relative, expected in kg_manifest['files'].items():
    assert hashlib.sha256((kg_build / relative).read_bytes()).hexdigest() == expected

kg_nodes = [r for path in sorted((kg_build/'nodes').glob('*.csv')) for r in csv.DictReader(path.open())]
kg_edges = list(csv.DictReader((kg_build/'relationships/relationships.csv').open()))
assert len(kg_nodes) == kg_manifest['counts']['nodes'] == 249
assert len(kg_edges) == kg_manifest['counts']['edges'] == 418
source_counts = Counter()
for row in kg_nodes:
    data = json.loads(row['properties_json'])
    source = data.get('source_record', {}).get('source_system')
    if source:
        source_counts[(source, row['type'])] += 1
node_counts = Counter(row['type'] for row in kg_nodes)

# Pin the exact baseline audit artifacts used for this checklist.
audit_dirs = {
    'DSLD': 'dsld/dsld-baseline-audit-v1-1487bf52f8e90a1f',
    'PubChem': 'pubchem/pubchem-baseline-audit-v1-ee86973dbcbcc97b',
    'RxNorm': 'rxnorm/rxnorm-baseline-audit-v1-bb5c0ce9a6faee14',
    'MeSH': 'mesh/mesh-baseline-audit-v1-be204bc83020940e',
    'NatMed': 'natmedpro/natmed-baseline-audit-v1-b548c344b8c0a9b7',
    'Reddit': 'reddit/reddit-baseline-audit-v1-058aea355a5bb848',
}
coverage_evidence = []
for source, relative in audit_dirs.items():
    directory = kg_root / 'data/processed' / relative
    for filename in ['manifest.json', 'audit.json']:
        path = directory / filename
        json.loads(path.read_bytes())
        coverage_evidence.append({'source': source, 'artifact': str(path.relative_to(kg_root)),
                                  'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
# This inventories saved audit evidence; it does not rerun the baseline audits.
coverage = [
 {'dataset':'DSLD', 'saved_scope':'400 labels', 'current_graph':f"{source_counts[('dsld','ProductLabel')]} labels", 'completed':'Full baseline/quantity/name audits; 400-label inventory; provisional fields', 'preparation':'Pilot-only normalization and graph export', 'next_work':'Batch all selected fields; serving/NP conflicts; product identity review'},
 {'dataset':'PubChem', 'saved_scope':'239 compounds / 739 queries', 'current_graph':f"{source_counts[('pubchem','ChemicalCompound')]} compounds", 'completed':'Full baseline audit; query lineage; provisional fields', 'preparation':'Selected candidate review and mappings only', 'next_work':'Review remaining candidates, salts/hydrates and class-versus-compound scope'},
 {'dataset':'RxNorm', 'saved_scope':'287 concepts / 739 queries', 'current_graph':f"{source_counts[('rxnorm','TerminologyConcept')]} concepts", 'completed':'Full baseline audit; exact/normalized search distinctions; provisional fields', 'preparation':'Selected candidate review and mappings only', 'next_work':'Review remaining name/form matches; preserve IN/PIN scope and ambiguous results'},
 {'dataset':'MeSH', 'saved_scope':'247 matched records / 739 queries', 'current_graph':f"{source_counts[('mesh','TerminologyConcept')]} concepts", 'completed':'Full baseline audit; term/concept/record checks; provisional fields', 'preparation':'Selected concept links; hierarchy fixture in preprocessing pilot', 'next_work':'Expand term/concept scope review and selected hierarchy/reference exports'},
 {'dataset':'NatMed', 'saved_scope':'10 monographs / 1,269 pending annotations', 'current_graph':f"{source_counts[('natmedpro','Monograph')]} monographs / {node_counts['ClinicalClaim']} pending claims", 'completed':'Full baseline/schema/span audit; provisional fields', 'preparation':'One Ashwagandha + ten Creatine claims exported', 'next_work':'Expand selected claim types with qualifiers/citations; retain extraction gaps and clinical review'},
 {'dataset':'Reddit', 'saved_scope':'4,880 records / 5,071 supplied passages', 'current_graph':f"{node_counts['RedditComment']} comments", 'completed':'Full baseline/context/passage audit; provisional fields', 'preparation':'Three-comment pilot; direct versus inherited-topic checks', 'next_work':'Batch record/text/parent/topic processing; relevance and duplicate review; avoid inferred health effects'},
]
coverage_frame = pd.DataFrame(coverage)
print(coverage_frame.to_string(index=False))
print('\nSaved evidence inventory:')
print(pd.DataFrame(coverage_evidence).to_string(index=False))
print('\nGraph manifest SHA256:', hashlib.sha256((kg_build/'manifest.json').read_bytes()).hexdigest())
print('No transformed data or new graph build written by this notebook.')


dataset                               saved_scope                    current_graph                                                                     completed                                                      preparation                                                                                                next_work
   DSLD                                400 labels                         7 labels   Full baseline/quantity/name audits; 400-label inventory; provisional fields                        Pilot-only normalization and graph export                                 Batch all selected fields; serving/NP conflicts; product identity review
PubChem               239 compounds / 739 queries                      2 compounds                        Full baseline audit; query lineage; provisional fields                      Selected candidate review and mappings only                              Review remaining candidates, salts/hydrates and class-versus-compound scope
 RxNorm

### Cross-dataset preparation checklist

| Area | Current status | Required before freezing the expanded KG |
|---|---|---|
| Audits and POC field selection | Complete baseline audits; field selections remain provisional | Confirm selected export fields and documented exclusions |
| Shared schema/source mappings | Designed; contract 0.1.0 implemented for the bounded pilot | Consolidate later graph additions into a versioned construction/export contract |
| Normalization | Supported mass comparison and preservation rules exercised in pilot | Apply to full selected scope; retain conflicts and unknowns |
| Mapping | 40 pilot candidates reviewed: 24 High, 8 Medium, 8 Low | Review remaining candidates; activate only supported relation types |
| Product/ingredient identity | Source occurrences retained; no canonical merges | Record supported equivalence decisions or explicitly keep identities source-scoped |
| Deduplication | Replay/version behavior tested; some shared excerpt reuse | Apply evidence-aware rules without collapsing populations, formulations or labels |
| Provenance/references | Pilot source pointers, hashes and spans checked | Resolve or quarantine required references across expanded scope |
| Clinical/Reddit interpretation | Pending claims and discussion roles preserved | Maintain these boundaries across all exported records |
| Graph validation | Pilot integrity/path checks and four fixed answer examples | Validate expanded counts, endpoints, mappings, coverage and exclusions |
| Dataset freeze | Versioned pilot builds exist | Produce an agreed POC build with a manifest, limits and reproducible workflow |

### Known unresolved items to carry forward

- BetterYou label 257287: 52 mL versus 0.52 ml conflict; no correction or affected dose calculation.
- NP quantities, unresolved salt/hydration details and ambiguous identities remain unknown/pending.
- Similar product names and equivalent quantities are insufficient for merging labels.
- NatMed clinical approval remains pending. Upstream extraction issues and missing bibliography references are not resolved by structural validation.
- Reddit inherited topics are not direct mentions; missing parent records must remain explicit until loaded and checked.
- Existing graph extensions have rules in scripts/notebooks beyond the original bounded contract. Consolidate them before scaling rather than treating the original contract as a finished full-data specification.

### Recommended next batch

Start **Step 2 with DSLD**: generalize the selected-field preprocessing from seven labels to the 400-label collection, preserving source-scoped identities and producing conflict/review queues. Keep cross-source mapping approvals and semantic merges as separate stages. Pin the updated transformation/export rules and batch checks before writing expanded outputs.

This checklist is a readiness review, not authorization to silently accept unresolved mappings or a claim that every raw record belongs in the final graph.


## Step 2 — Expand normalization

**DSLD batch complete. Terminology expansion is documented in Step 2B below; NatMed/Reddit are pending.** This run applies the selected-field rules to all 400 labels, not just the seven-label pilot. The current graph remains unchanged at 249 nodes and 418 relationships; full graph expansion belongs to later stages.

### Batch rules and outputs

- Reuses the tested DSLD extractor/table shapes from contract 0.1.0 under [DSLD batch specification v1](../schemas/dsld_normalization_batch_v1.json); the original pilot contract and artifacts are unchanged.
- Preserves every label, all 5,438 ingredient/form occurrences, 400 servings and 2,392 quantity entries. Selected directions, precautions, storage, other-ingredient text and source relationships are retained.
- Produces separate display/search names and exact decimal mass representations in mg where supported. Original names, values, operators and serving bases remain unchanged.
- NP quantities remain unknown; unsupported units are not converted. No activity/volume conversions, per-unit/daily-dose derivations, accepted mappings, canonical identities or semantic merges are produced.
- All 872 prior quantity-audit findings are retained individually with evidence pointers. Review-queue rows can aggregate several findings; they are not distinct source-error counts.
- Resolved label relationships refer to this 400-label collection. Out-of-sample targets remain unresolved; their existence elsewhere is not denied.

Run `supplementsrx_env/bin/python scripts/normalize_dsld_batch.py` from the project root. It writes an immutable, hash-versioned build under `data/processed/dsld/`; the same inputs/code reuse identical output.

### Conversion versus comparison eligibility

**1,487 quantities have a supported mg representation**, but only **1,412 are eligible for scoped comparison**. Another **75 require review**, and **905 have no supported mass representation** (705 NP entries, 199 unsupported-unit entries and one serving conflict). Eligibility still requires compatible ingredient identity, basis and operator; it does not mean records are duplicates or quantities are clinically recommended.


In [2]:
from pathlib import Path
import json, hashlib
from collections import Counter
import pandas as pd
batch_root = Path.cwd()
if not (batch_root / 'data').is_dir():
    batch_root = batch_root.parent
batch_output = batch_root / 'data/processed/dsld/dsld-normalized-v1-66adbe28c3b0344457fb'
batch_manifest = json.loads((batch_output / 'manifest.json').read_text())
for filename, expected in batch_manifest['files'].items():
    assert hashlib.sha256((batch_output / filename).read_bytes()).hexdigest() == expected
batch_report = json.loads((batch_output / 'validation_report.json').read_text())
print('Scope:', batch_report['scope'])
print('Counts:', {k: batch_report[k] for k in ['labels','occurrences','quantity_entries','servings','evidence_pointers_checked']})
print('Mass outcomes:', batch_report['mass_outcomes'])
print('Comparison eligibility:', batch_report['comparison_statuses'])
print('Blocked reasons:', batch_report['mass_block_reasons'])
print('Audit findings:', batch_report['audit_findings_by_code'])
print('Checks:', batch_report['checks'])
assert sum(batch_report['comparison_statuses'].values()) == batch_report['quantity_entries'] == 2392
assert all(batch_report['checks'].values())
assert batch_report['graph_updated'] is False
print('\nReview stages (overlapping issue rows):', batch_report['review_rows_by_stage'])


Scope: DSLD selected fields across all 400 labels; other five sources not expanded
Counts: {'labels': 400, 'occurrences': 5438, 'quantity_entries': 2392, 'servings': 400, 'evidence_pointers_checked': 12836}
Mass outcomes: {'blocked': 905, 'normalized': 1487}
Comparison eligibility: {'blocked': 905, 'eligible_for_scoped_comparison': 1412, 'pending_review': 75}
Blocked reasons: {'source_numeric_text_conflict': 1, 'unsupported_source_amount': 705, 'unsupported_unit': 199}
Audit findings: {'incomplete_daily_serving_range': 35, 'inequality_amount': 24, 'missing_quantity_serving_basis': 8, 'ml_note_numeric_disagreement': 1, 'np_unit_unresolved': 705, 'quantity_basis_outside_serving_range': 67, 'zero_amount_with_named_unit': 32}
Checks: {'betteryou_conflict_blocked': True, 'bpi_equivalent_mass_separate_labels': True, 'evidence_verified': True, 'np_not_zero': True, 'original_quantity_values_retained': True, 'raw_and_prior_artifacts_unchanged': True, 'recursive_source_coverage': True, 'table_sc

In [3]:
def batch_rows(name):
    return {r['id']: r for r in map(json.loads, (batch_output / (name + '.jsonl')).read_text().splitlines())}
batch_assertions = batch_rows('assertions')
batch_snapshots = batch_rows('snapshots')
batch_records = batch_rows('source_records')
batch_transforms = batch_rows('transformations')
batch_reviews = list(batch_rows('quantity_comparison_review').values())
examples = []
for review in batch_reviews:
    transformation = batch_transforms[review['transformation_id']]
    assertion = batch_assertions[review['assertion_id']]
    label_id = batch_records[batch_snapshots[assertion['snapshot_id']]['record_id']]['native_id']
    examples.append({'label_id': label_id,
                     'ingredient': assertion['original_fields']['ingredient_name'],
                     'original_quantity': transformation['original_value'],
                     'mass_representation': transformation['normalized_value'],
                     'comparison_status': review['status'],
                     'reason_codes': review['reason_codes']})
chosen = [r for r in examples if (r['label_id'] in ['25731','43261'] and r['ingredient']=='Creatine') or r['label_id']=='257287']
for reason in ['unsupported_source_amount','unsupported_unit','missing_or_invalid_basis','quantity_basis_outside_serving_range','zero_named_unit_needs_review']:
    example = next((r for r in examples if reason in r['reason_codes']), None)
    if example and example not in chosen:
        chosen.append(example)
for example in chosen:
    print(json.dumps(example, indent=2, ensure_ascii=False))
print('\nPer-row comparison-status total:', Counter(r['comparison_status'] for r in examples))


{
  "label_id": "43261",
  "ingredient": "Creatine",
  "original_quantity": {
    "dailyValueTargetGroup": [
      {
        "footnote": "Daily Value not established",
        "name": "Adults and children 4 or more years of age",
        "operator": null,
        "percent": null
      }
    ],
    "operator": "=",
    "quantity": 1.5,
    "servingSizeOrder": 1,
    "servingSizeQuantity": 2,
    "servingSizeUnit": "Capsule(s)",
    "unit": "Gram(s)"
  },
  "mass_representation": {
    "amount": "1500",
    "basis_quantity": 2,
    "basis_unit": "Capsule(s)",
    "operator": "=",
    "serving_order": 1,
    "unit": "mg"
  },
  "comparison_status": "eligible_for_scoped_comparison",
  "reason_codes": []
}
{
  "label_id": "25731",
  "ingredient": "Creatine",
  "original_quantity": {
    "dailyValueTargetGroup": [
      {
        "footnote": "Daily Value not established",
        "name": "Adults and children 4 or more years of age",
        "operator": null,
        "percent": null
      }
 

### Validation and remaining Step 2 work

- All 400 labels and recursive source paths reconcile with outputs; every exported evidence pointer was checked (12,836).
- BPI 1.5 g and 1,500 mg retain the same two-capsule basis without merging labels. BetterYou's serving conflict remains blocked; NP is never converted to zero.
- Table types, foreign keys, source references, original quantities and raw/prior artifact hashes pass validation. Identical replay was checked. The full test suite passes **85 tests**, including ten new batch tests.
- This completes the first Step 2 batch for **DSLD selected fields**. At the end of this DSLD batch, the other sources remained at pilot scope; Step 2B below records the subsequent terminology expansion.
- Next: expand source-specific terminology preparation, keeping raw candidates separate from Step 3 mapping decisions. NatMed/Reddit processing follows with original text, context and review states preserved.

The Step 1 coverage table above is the **pre-batch baseline**. Its DSLD “pilot-only normalization” entry is superseded by this section; DSLD graph coverage is still seven labels until graph construction is expanded.


### Step 2B — Collection-wide terminology preparation

**Completed for PubChem, RxNorm and MeSH.** These sources are prepared together for consistent query/DSLD linkage, with separate output folders. This batch does not activate mappings or update the graph.

- Each source retains all **739 queries** and **3,391 originating DSLD ingredient/form occurrences**, linked to the prepared 400-label batch.
- PubChem: **239 compounds**, original chemical properties and all **46,426 synonym entries**. Three missing IUPAC names remain missing.
- RxNorm: **287 concepts**, basic properties, selected names/source-vocabulary metadata and UNII/DrugBank/SNOMEDCT code candidates. Exact and normalized searches remain distinguishable. Unselected bulk properties remain in raw data.
- MeSH: **247 matched records**, **2,360 described subjects**, **24,792 distinct typed RDF statements** and **276 term–concept–record resolution bindings**. Repeated statement evidence is retained; typed literals, languages and URIs are not conflated.
- Normalized display/search strings are separate from source names and IDs. Missing/null/empty fields and original types are retained; molecular weight is not a dosage.
- All **895 query-to-target candidates** remain `candidate_unspecified` / `needs_review` with null confidence. These are query-level links, not occurrence-expanded mappings. Step 3 will make confidence/identity decisions; prior pilot decisions remain in their own unchanged artifacts.
- No-match queries remain explicit: PubChem **511**, RxNorm **426**, MeSH **469**. Ambiguous-query counts are **7**, **31** and **6** respectively. No-match reflects the saved search policy, not universal absence.

Specification: [terminology preparation v1](../schemas/terminology_preparation_batch_v1.json). Run `supplementsrx_env/bin/python scripts/prepare_terminology_batch.py` from the project root.

Each source folder contains records, properties, queries, query origins, candidates and review queues. MeSH additionally retains subjects, typed statements, term resolution and external references. Source pointers resolve into the hash-pinned raw collection; DSLD occurrence/evidence IDs resolve into the declared DSLD batch dependency. Thus each folder is source-separated, not an entirely standalone graph.


In [4]:
from pathlib import Path
import json, hashlib
import pandas as pd
term_root = Path.cwd()
if not (term_root / 'data').is_dir():
    term_root = term_root.parent
term_output = term_root / 'data/processed/terminology/terminology-prepared-v1-f8afbddf6af7037ac98e'
term_manifest = json.loads((term_output / 'manifest.json').read_text())
for filename, expected in term_manifest['files'].items():
    assert hashlib.sha256((term_output / filename).read_bytes()).hexdigest() == expected
term_report = json.loads((term_output / 'validation_report.json').read_text())
summary = []
for source, report in sorted(term_report['sources'].items()):
    summary.append({'source':source,'records':report['rows']['records'],
                    'queries':report['rows']['queries'],'DSLD occurrences':report['rows']['query_origins'],
                    'candidate links':report['rows']['candidates'],
                    'no match':report['query_outcomes']['no_candidate'],
                    'ambiguous queries':report['query_outcomes']['ambiguous_candidates'],
                    'verified source pointers':report['verified_source_pointers']})
print(pd.DataFrame(summary).to_string(index=False))
print('\nChecks:', term_report['checks'])
print('Graph updated:', term_report['graph_updated'])
assert sum(row['candidate links'] for row in summary) == 895
assert all(row['queries'] == 739 and row['DSLD occurrences'] == 3391 for row in summary)


 source  records  queries  DSLD occurrences  candidate links  no match  ambiguous queries  verified source pointers
   mesh      247      739              3391              276       469                  6                     31076
pubchem      239      739              3391              272       511                  7                     53443
 rxnorm      287      739              3391              347       426                 31                     10165

Checks: {'all_candidates_preserved': True, 'dsld_occurrences_verified': True, 'mesh_typed_values_preserved': True, 'query_inventory_consistent': True, 'source_hashes': True, 'source_pointers_resolve': True}
Graph updated: False


In [5]:
def term_rows(source, table):
    return [json.loads(line) for line in (term_output/source/(table+'.jsonl')).read_text().splitlines()]
for source in ['pubchem','rxnorm','mesh']:
    print('\nSOURCE:', source)
    queries = term_rows(source,'queries')
    for name in ['Creatine HCl','Magnesium Citrate','Ashwagandha']:
        selected = [q for q in queries if q['original_query'] == name]
        for q in selected:
            print(json.dumps({k:q[k] for k in ['original_query','match_method','outcome','candidate_native_ids']}, ensure_ascii=False))
    candidates = term_rows(source,'candidates')
    assert all(c['confidence'] is None and c['review_state']=='needs_review' for c in candidates)
missing_iupac = [r['native_id'] for r in term_rows('pubchem','records') if r['field_states']['IUPACName'] != 'present']
print('\nPubChem records retaining missing IUPAC name:', missing_iupac)
assert len(missing_iupac) == 3
mesh_references = term_rows('mesh','external_references')
print('MeSH object-URI references without descriptions:', len(mesh_references))
print('Distinct external target URIs:', len({r['target_uri'] for r in mesh_references}))
print('These are references outside the described saved subgraphs, not fabricated concepts or collection failures.')



SOURCE: pubchem
{"original_query": "Creatine HCl", "match_method": "saved_name_lookup", "outcome": "single_candidate", "candidate_native_ids": ["134732"]}
{"original_query": "Magnesium Citrate", "match_method": "saved_name_lookup", "outcome": "single_candidate", "candidate_native_ids": ["6099959"]}
{"original_query": "Ashwagandha", "match_method": "saved_name_lookup", "outcome": "no_candidate", "candidate_native_ids": []}

SOURCE: rxnorm
{"original_query": "Creatine HCl", "match_method": "normalized", "outcome": "ambiguous_candidates", "candidate_native_ids": ["2907", "1310467"]}
{"original_query": "Magnesium Citrate", "match_method": "exact", "outcome": "single_candidate", "candidate_native_ids": ["52356"]}
{"original_query": "Ashwagandha", "match_method": null, "outcome": "no_candidate", "candidate_native_ids": []}

SOURCE: mesh
{"original_query": "Creatine HCl", "match_method": "exact_entry_term", "outcome": "no_candidate", "candidate_native_ids": []}
{"original_query": "Magnesium 

### Step 2B validation and next source batches

- Input hashes, shared query inventory, all DSLD occurrence identities/source hashes, target IDs and source pointers passed validation.
- Tests preserve RxNorm's incompatible Creatine HCl candidates for later review, reject unsuccessful/contradictory lookup statuses, and distinguish RDF language-tagged literals.
- The full suite passes **94 tests**, including nine terminology-batch tests. Identical output replay is checked; raw collections, earlier preparation outputs and the graph remain unchanged.
- **Step 2 status:** DSLD and all three terminology sources now have collection-wide selected-field preparation. **NatMed and Reddit still require their full selected-field batches.** Semantic mapping and deduplication decisions remain Steps 3–4.
- Next: prepare NatMed's saved monographs/annotations with source support, original context and pending review states; then expand Reddit's records, topic/context and parent-reference handling.


### Step 2C — Collection-wide NatMed and Reddit preparation

**Completed for both saved collections.** With the DSLD and terminology batches, all six sources now have collection-wide selected-field preparation. This completes Step 2's batch preparation scope, not identity resolution, semantic deduplication, clinical approval or full graph construction.

**NatMed:** 10 monographs, 1,269 pending annotations and 1,179 passages. Original fields/text, population/formulation unknowns, dose context and citations are preserved. Schema, source hashes, excerpt spans and all 2,923 field-support anchors were checked. The 769 anchors referring to an earlier effectiveness heading remain explicitly marked. All 10 HTML bindings and bibliography documents are verified against saved hashes.

**Reddit:** 4,880 original records (287 posts, 4,593 comments) and 5,071 passages. Original text, timestamps, URLs, null scores and record-level mention scope remain unchanged. Per-topic checks found 1,672 literal-alias associations and 6,310 inherited-thread associations. Literal hits include the source field and character offsets; they do not establish personal use or health effects. Inherited context is never upgraded automatically.

**References:** 9,261 parent/thread references resolve within the full Reddit collection; 113 references point outside it; 99 parent IDs are missing; 287 post-parent slots are not applicable. These are reference-row counts, not distinct records or missing targets. No posts or parent IDs were fabricated.

**Issue handling:** duplicate groups remain review metadata, not merges. NatMed's 49 upstream extraction issues and 55 reported missing bibliography numbers remain visible, not corrected. All 1,269 clinical annotations remain pending; no mapping confidence is assigned in this stage.

Specification: [NatMed/Reddit preparation v1](../schemas/natmed_reddit_preparation_batch_v1.json). Run `supplementsrx_env/bin/python scripts/prepare_natmed_reddit_batch.py` from the project root. Outputs are separated into `natmedpro/` and `reddit/`, with source pointers, hashes, review queues and a shared manifest.


In [6]:
from pathlib import Path
import json, hashlib
import pandas as pd
imported_root = Path.cwd()
if not (imported_root / 'data').is_dir():
    imported_root = imported_root.parent
imported_output = imported_root / 'data/processed/imported/natmed-reddit-prepared-v1-b85118347d2f8de1a1ea'
imported_manifest = json.loads((imported_output / 'manifest.json').read_text())
for filename, expected in imported_manifest['files'].items():
    assert hashlib.sha256((imported_output / filename).read_bytes()).hexdigest() == expected
imported_report = json.loads((imported_output / 'validation_report.json').read_text())
for source, report in imported_report['sources'].items():
    print('\nSOURCE:', source)
    print(json.dumps(report['summary'], indent=2))
    print('Prepared tables:', report['table_counts'])
    print('Checks:', report['checks'])
print('Graph updated:', imported_report['graph_updated'])
assert imported_report['clinical_approvals'] == 0
assert imported_report['semantic_merges'] == 0
assert imported_report['sources']['natmedpro']['summary']['pending_clinical_claims'] == 1269



SOURCE: natmedpro
{
  "claim_types": {
    "adverse_effect": 120,
    "dose": 85,
    "effectiveness": 809,
    "interaction": 148,
    "safety": 107
  },
  "claims": 1269,
  "earlier_heading_anchors": 769,
  "field_support_anchors": 2923,
  "passages": 1179,
  "pending_clinical_claims": 1269,
  "records": 10,
  "upstream_extraction_issues": {
    "inline_condition_heading": 2,
    "no_structured_fields": 6,
    "placeholder_not_negative_evidence": 7,
    "section_not_in_html": 1,
    "section_qualification": 10,
    "unquantified_dose_statement": 5,
    "unresolved_dose_duration": 1,
    "unresolved_dose_normalization": 9,
    "unresolved_dose_population": 8
  },
  "upstream_missing_bibliography_numbers": 55
}
Prepared tables: {'claims': 1269, 'field_support': 2923, 'html_bindings': 10, 'passages': 1179, 'records': 10, 'reference_documents': 10, 'references': 0, 'review_queue': 1279, 'topic_associations': 10}
Checks: {'clinical_approvals': 0, 'complete_counts': True, 'foreign_keys_re

In [7]:
def imported_rows(source, table):
    return [json.loads(line) for line in (imported_output/source/(table+'.jsonl')).read_text().splitlines()]
claims = imported_rows('natmedpro','claims')
fields = imported_rows('natmedpro','field_support')
anchor = next(f for f in fields if f['anchor_scope']=='earlier_section_heading')
claim = next(c for c in claims if c['id']==anchor['claim_id'])
print('EXAMPLE: earlier rating heading, preserved independently of claim excerpt')
print(json.dumps({'claim_id':claim['id'],'ingredient':claim['original_fields']['ingredient'],
                  'population':claim['original_fields']['population'],'indication':claim['original_fields']['indication'],
                  'review_state':claim['review_state'],'source_excerpt_span':claim['source_span'],
                  'rating_anchor':anchor}, indent=2))
reddit_records = {r['id']:r for r in imported_rows('reddit','records')}
reddit_topics = imported_rows('reddit','topic_associations')
print('\nEXAMPLES: explicit alias versus inherited context')
for status in ['literal_alias_in_record','inherited_thread_context']:
    association = next(t for t in reddit_topics if t['topic_status']==status)
    original = reddit_records[association['record_id']]['original_fields']
    print(json.dumps({'topic':association['topic'],'status':status,'original_scope':association['original_mention_scope'],
                      'literal_hits':association['literal_hits'],'original_text':original['text'],
                      'source_url':original['source_url']}, indent=2, ensure_ascii=False))
print('\nEXAMPLES: unresolved reference states')
references = imported_rows('reddit','references')
for status in ['missing_identifier','outside_saved_collection','not_applicable']:
    print(json.dumps(next(r for r in references if r['status']==status), indent=2))
assert all(c['review_state']=='pending' and c['confidence'] is None for c in claims)


EXAMPLE: earlier rating heading, preserved independently of claim excerpt
{
  "claim_id": "claim:fa419545ee8b346d702d891654ae7276f5fa981bd569109cad680f22548317f6",
  "ingredient": {
    "formulation": null,
    "name": "Vitamin D",
    "strain": null
  },
  "population": null,
  "indication": "Urinary incontinence.",
  "review_state": "pending",
  "source_excerpt_span": {
    "coordinate_system": "unicode_codepoints",
    "end": 154985,
    "file": "data/raw/natmedpro/records.jsonl",
    "pointer": "/7/sections/6/text",
    "sha256": "b413d5ff9051e5ffdbd5c44b804aaf6b9a17f4cdf449e7f60937a7e901b4560c",
    "start": 153885
  },
  "rating_anchor": {
    "anchor_scope": "earlier_section_heading",
    "claim_id": "claim:fa419545ee8b346d702d891654ae7276f5fa981bd569109cad680f22548317f6",
    "field": "effectiveness_rating",
    "id": "prepared_claim_field:002d99e715611ce306bce050c7bf15307ab3927c0f46745c472d6e7821462c9e",
    "original_anchor_provenance": {
      "file": "data/raw/natmedpro/ups

### Step 2 completion checkpoint

| Dataset | Prepared scope | Still outside Step 2 |
|---|---|---|
| DSLD | All 400 labels and 2,392 quantity entries | Conflict resolution, canonical grouping and full graph export |
| PubChem | All 239 compounds and 739 queries | Ingredient/material mapping decisions |
| RxNorm | All 287 concepts and 739 queries | Name/form scope decisions |
| MeSH | All 247 candidate records, 2,360 described subjects and saved RDF | Semantic mapping decisions and graph projection choices |
| NatMed | All 10 monographs, 1,269 pending annotations and 1,179 passages | Clinical review, extraction/reference gaps and graph expansion |
| Reddit | All 4,880 records and 5,071 passages | Semantic relevance/deduplication and graph expansion |

All **105 tests pass**, including eleven new NatMed/Reddit tests. Source fields and passage text remain unchanged; full-source references and field support are validated. Repeated execution produces identical outputs. No graph links were added; the existing graph remains at 249 nodes and 418 relationships.

The earlier Step 1 and batch-local status notes are historical snapshots. This checkpoint records current coverage. “Prepared” means selected-field preservation, structure, provenance and supported representations—not that every ambiguity has been resolved or every record is suitable for an answer.

**Next: Step 3 — review ingredient mappings across the prepared sources using High/Medium/Low rules and explicit relation types.** Keep unresolved identities pending rather than manufacturing equivalence.


## Step 3 — Ingredient mapping decisions

**Collection-wide rule-based triage completed; unresolved mappings remain pending.** This is not exhaustive human identity validation, clinical approval or graph expansion. Every saved terminology query-candidate/origin combination has a decision or explicit no-candidate outcome.

### Rules applied

- **M1 — Existing evidence review:** retain earlier decisions only for the same source occurrence/target and pinned inputs. Preserved all 40 pilot decisions, 17 additional prior scope decisions, two reviewed category proposals and four existing NatMed subject links.
- **M2 — High concept mapping:** one active RxNorm/MeSH candidate, matching preferred name after case/whitespace normalization, and no unresolved botanical/blend/preparation scope. Relation is `denotes_concept`, never an inferred exact-material match.
- **M3 — Medium:** unreviewed synonym bridges, multiple candidates, inactive/unclear target status, preparation details or unreviewed PubChem material specificity. Keep pending.
- **M4 — Low for exact identity:** explicit hydrochloride-versus-hydrate conflict or an undeclared allergenic-extract preparation. Exclude exact matching; do not invent an alternative relation.
- **M5 — No candidate:** retain unresolved with null confidence. No-match is not Low confidence or proof of absence.
- **M6 — NatMed:** exact general ingredient names support only `related_ingredient_subject`. Group-only/qualified preparations remain Medium unless a previous evidence-backed review supplies the bridge. No product-specific safety or dose applicability is implied.
- **M7 — Reddit/source topics:** retain topic-context relationships separately from ingredient identities; inherited context remains inherited.

Specification: [mapping decisions v1](../schemas/ingredient_mapping_decisions_v1.json). Reproduce with `supplementsrx_env/bin/python scripts/decide_ingredient_mappings.py`.

### Counts and meaning

The 895 query-to-target candidates expand to **6,735 occurrence-level decisions**: **2,684 High, 4,003 Medium and 48 Low**. A target reached from different labels remains a separate mapping decision, not a duplicate to delete.

High consists of **2,677 concept links and seven previously reviewed exact named-form links**. No new exact-material matches are inferred from names. Two additional reviewed category proposals are stored separately and not counted among the 6,735 lookup-derived decisions.

Another **4,207 occurrence/source lookups** have no candidate and remain unresolved. These counts span three terminology sources and are not counts of unique supplements or labels.

NatMed has **353 High and 163 Medium general-subject proposals**. Another 1,823 DSLD ingredient occurrences have no saved title/group bridge to a NatMed monograph; most are co-ingredients outside the ten monographs. The **7,992 NatMed/Reddit topic associations** remain source context, not chemical identity links.


In [8]:
from pathlib import Path
import json, hashlib
import pandas as pd
mapping_root = Path.cwd()
if not (mapping_root / 'data').is_dir():
    mapping_root = mapping_root.parent
mapping_output = mapping_root / 'data/processed/identity/ingredient-decisions-v1-1e02682e87ad7456218b'
mapping_manifest = json.loads((mapping_output / 'manifest.json').read_text())
for filename, expected in mapping_manifest['files'].items():
    assert hashlib.sha256((mapping_output / filename).read_bytes()).hexdigest() == expected
mapping_report = json.loads((mapping_output / 'validation_report.json').read_text())
def decision_rows(name):
    return [json.loads(line) for line in (mapping_output / (name+'.jsonl')).read_text().splitlines()]
mapping_rows = pd.DataFrame(decision_rows('terminology_decisions'))
print('Coverage:')
print(pd.DataFrame(mapping_report['coverage']).to_string(index=False))
print('\nConfidence by target source:')
print(pd.crosstab(mapping_rows.target_source, mapping_rows.confidence).to_string())
print('\nHigh relationship types:')
print(mapping_rows.loc[mapping_rows.confidence=='High','proposed_relation'].value_counts().to_string())
print('\nNatMed subject coverage:')
print(pd.DataFrame(mapping_report['natmed_monograph_coverage']).to_string(index=False))
print('\nValidation:', mapping_report['checks'])
assert len(mapping_rows)==6735
assert mapping_report['prior_pilot_decisions_preserved']==40
assert mapping_report['prior_natmed_subject_links_preserved']==4
assert mapping_report['graph_updated'] is False


Coverage:
 no_candidate_occurrences  occurrence_candidate_decisions  queries  query_candidates  source
                     1573                            2478      739               272 pubchem
                     1310                            2179      739               347  rxnorm
                     1324                            2078      739               276    mesh

Confidence by target source:
confidence     High  Low  Medium
target_source                   
mesh           1228    0     850
pubchem           4    3    2471
rxnorm         1452   45     682

High relationship types:
proposed_relation
denotes_concept         2677
exact_material_match       7

NatMed subject coverage:
                    counts                    monograph_id                     title
{'High': 13, 'Medium': 46} natmed:0d296c0daa1cccb456da568c               Ashwagandha
              {'High': 56} natmed:0e9f2262886c394a16e6a7b6               Vitamin B12
{'High': 41, 'Medium': 29} natmed:15e8a6

In [9]:
# Inspect relation-specific decisions rather than interpreting confidence alone.
examples = []
for name, confidence in [('Magnesium','High'),('Magnesium Citrate','Medium'),('Creatine HCl','Low'),('Acacia Gum','Low')]:
    candidates = mapping_rows[(mapping_rows.original_name==name) & (mapping_rows.confidence==confidence)]
    if not candidates.empty:
        examples.append(candidates.iloc[0].to_dict())
for example in examples:
    print(json.dumps({k:example[k] for k in ['label_id','original_name','target_source','target_native_id','target_name','confidence','proposed_relation','assessed_relation','rule','reason','action','prior_review','evidence']},indent=2))
print('\nExample no-candidate outcome (not a Low-confidence mapping):')
print(json.dumps(decision_rows('no_candidate_queries')[0],indent=2))
print('\nNatMed subject proposals are not clinical approvals:')
subjects = decision_rows('natmed_subject_decisions')
for confidence in ['High','Medium']:
    print(json.dumps(next(r for r in subjects if r['confidence']==confidence),indent=2))
assert all(r['rule']=='M1' for r in decision_rows('terminology_decisions') if r['proposed_relation']=='exact_material_match')
assert all(r['confidence'] is None for r in decision_rows('no_candidate_queries'))
assert all(not r['identity_mapping'] and not r['clinical_inference'] for r in decision_rows('source_topic_context'))


{
  "label_id": "30599",
  "original_name": "Magnesium",
  "target_source": "mesh",
  "target_native_id": "http://id.nlm.nih.gov/mesh/D008274",
  "target_name": "Magnesium",
  "confidence": "High",
  "proposed_relation": "denotes_concept",
  "assessed_relation": "denotes_concept",
  "rule": "M2",
  "reason": "Single active target has the same explicit preferred name; link is limited to the named concept, not exact material or product identity.",
  "action": "eligible_for_graph_construction",
  "prior_review": null,
  "evidence": {
    "candidate_provenance": [
      {
        "pointer": "/search_results/magnesium",
        "source": "mesh"
      }
    ],
    "dsld_evidence_ids": [
      "evidence:a22850493bd8ed299ea32b501cafb4b471047d02a59e0079b68c0aa6f889d02f"
    ],
    "query_origin_id": "prepared_origin:cedb19e242d551c9369096ac0c25a70b6977f2aeeb98fbee549c81b98b6ed9dd",
    "target_record_id": "prepared_record:26b1966478d6f4b32a1a98bf8e93d7113cb14aa463bc8f1699fa342b871f9585",
    "t

### Step 3 outcome and limits

- All saved candidate/origin combinations are assessed; original query, target and DSLD evidence references are retained. Prior pilot decisions are not silently discarded or generalized to unrelated occurrences.
- High means **eligible for later construction at the stated relationship level**. `rule_assessed` is not a claim of human review. Medium and no-candidate cases remain pending; Low excludes exact matching only.
- Scope safeguards, prior-decision preservation and no-match handling are tested. The full suite passes **118 tests**, including thirteen mapping tests; identical replay is verified.
- Raw/prepared inputs and the current graph remain unchanged. No canonical merges, clinical approvals or new graph edges were created.
- This completes Step 3's initial rule-based decision inventory, not resolution of every ambiguity. A large Medium queue is expected because name similarity does not settle material identity.

**Next: Step 4 — identity and duplicate review.** Start from the eligible relationship types and existing repeated-label audits. Keep source-scoped records where equivalence is unproven; do not treat two labels connected to the same concept as duplicate products.


## Step 4 — Resolve identities and duplicates

**Completed as a conservative rule-based review; unresolved product equivalence remains explicit.** The batch checks all 70 existing DSLD repeated-name groups and recomputes the NatMed/Reddit duplicate groups against prepared records. It does not claim an exhaustive search for equivalent products with different names.

- Keep every source label, claim, post and passage ID. Same names, ingredient cores, quantities or terminology targets do not prove entity identity.
- Keep direct DSLD `labelRelationships` as attributed source annotations, with their original type and pointer. Do not infer transitive equivalence.
- Share only exact text representations, with separate source bindings. Repeated excerpts do not become independent corroboration; repeated Reddit text does not establish one author or experience.
- Preserve all pending clinical reviews. No canonical merges or graph changes are made.

Rules: `schemas/identity_deduplication_decisions_v1.json`. Reproducible implementation: `scripts/review_identity_duplicates.py`.


In [10]:
from pathlib import Path
import sys, json
identity_root = Path.cwd()
if not (identity_root / 'scripts').is_dir():
    identity_root = identity_root.parent
if str(identity_root / 'scripts') not in sys.path:
    sys.path.insert(0, str(identity_root / 'scripts'))
from review_identity_duplicates import run
identity_output, identity_report = run(identity_root)


data/processed/identity/identity-duplicates-v1-3aa86ab20b94fd928b6b
{
  "dsld_groups": 70,
  "dsld_labels_in_groups": 206,
  "dsld_pairs": 304,
  "dsld_groups_same_compared_core": 27,
  "dsld_pairs_with_direct_source_links": 258,
  "dsld_payload_duplicate_pairs": 0,
  "duplicate_groups": {
    "natmedpro": {
      "record_text": 0,
      "passage_text": 5,
      "claim_text": 13,
      "claim_payload_except_id": 0,
      "claim_source_span_and_type": 12
    },
    "reddit": {
      "record_text": 24,
      "passage_text": 25,
      "record_payload_except_identity": 0
    }
  },
  "exact_text_blobs": 50,
  "text_bindings": 557,
  "entity_merges": 0,
  "graph_updated": false,
  "clinical_approvals": 0,
  "checks": {
    "saved_duplicate_groups_recomputed": true,
    "source_label_hashes_verified": true,
    "prepared_manifest_verified": true,
    "unique_source_ids": true
  },
  "limitations": [
    "DSLD review covers the 70 existing repeated-name groups, not exhaustive cross-name produ

In [11]:
def identity_rows(name):
    return [json.loads(line) for line in (identity_output / (name + '.jsonl')).read_text().splitlines()]
identity_pairs = identity_rows('dsld_pair_decisions')
print('DSLD example: BPI Creatine Alkaline labels 25731 and 43261')
example = next(p for p in identity_pairs if set(p['label_ids']) == {'25731', '43261'})
print(json.dumps({k: example[k] for k in ['label_ids', 'action', 'direct_source_relationships', 'differing_fields', 'audit_flags', 'canonical_merge']}, indent=2))
print('The earlier quantity audit found 1,500 mg and 1.5 g equivalent per two capsules.')
print('This permits an equivalent quantity comparison; differing other ingredients still prevent a whole-label merge.')


DSLD example: BPI Creatine Alkaline labels 25731 and 43261
{
  "label_ids": [
    "25731",
    "43261"
  ],
  "action": "retain_separate_source_related_labels",
  "direct_source_relationships": [
    {
      "from_label_id": "25731",
      "pointer": "/labelRelationships/0",
      "source_type": "Package size difference, same product",
      "to_label_id": "43261"
    },
    {
      "from_label_id": "43261",
      "pointer": "/labelRelationships/0",
      "source_type": "Package size difference, same product",
      "to_label_id": "25731"
    }
  ],
  "differing_fields": [
    "contacts",
    "entryDate",
    "events",
    "ingredientRows",
    "labelRelationships",
    "netContents",
    "offMarket",
    "otheringredients",
    "productVersionCode",
    "statements"
  ],
  "audit_flags": [
    "amount_or_basis_difference",
    "other_ingredients_raw_difference"
  ],
  "canonical_merge": false
}
The earlier quantity audit found 1,500 mg and 1.5 g equivalent per two capsules.
This permi

In [12]:
duplicate_decisions = identity_rows('content_duplicate_decisions')
print('NatMed example: same claim text, different dose fields')
claim_example = next(d for d in duplicate_decisions if d['source'] == 'natmedpro' and d['category'] == 'claim_text' and 'dose' in d['context_difference_fields'])
print(json.dumps({k: claim_example[k] for k in ['member_ids', 'context_difference_fields', 'action', 'canonical_merge']}, indent=2))
print('Reddit example: repeated text across different threads')
reddit_example = next(d for d in duplicate_decisions if d['source'] == 'reddit' and d['category'] == 'record_text' and 'thread_id' in d['context_difference_fields'])
print(json.dumps(dict(member_count=len(reddit_example['member_ids']), example_ids=reddit_example['member_ids'][:3], context_difference_fields=reddit_example['context_difference_fields'], action=reddit_example['action']), indent=2))
print('Full decisions and provenance:', identity_output)


NatMed example: same claim text, different dose fields
{
  "member_ids": [
    "claim:02bf195e731466380a569dbbd587364af3e4c1e62ed2c96d85ce9863770321d6",
    "claim:06fcf4bb7409593c7562fbda7db9313963834e689e63d02b470e724a77bf2661",
    "claim:0aab51af18a01e28b860e56ad90527521b13ebdf976e29d451c7923ccb3aff28",
    "claim:3ae9fc5d22b40e61a84f77e079cc6b6fe22da95d1f5b318a625811637aae4853",
    "claim:5a7aa14d50375ab89ec14b1969c1bfd6785e2e6d1d7850bf8bc46084e8289772",
    "claim:7b3305d530b755f51e4ce9740912ce8f072c45b8b1192603fefb90781718100a",
    "claim:8b42fe82def9905f60f2595a5685b06c069f91b321062005a92d22b505402a93",
    "claim:b55a6dd08085f70c735ac0fb0d33f79a84c64afeeb109605eb064af2a6a4d4c1",
    "claim:d323e70054216d2e8293cfefaf2cf274325265a375f98a7d9d19bd0e4199eaf0",
    "claim:d72ad7b6e8302010a08d8865432c906e9b284a4e7e735b0bdf6debaf16c2aa6f",
    "claim:dcf684dbf28860218d7bd0236c4b1003848e1ea4920888d1a1216b99979817dc"
  ],
  "context_difference_fields": [
    "dose",
    "field_support

### Step 4 checkpoint

- **DSLD:** 70 groups, 206 labels, 304 pairs reviewed. 27 groups match the earlier compared ingredient/serving core. 258 pairs have direct source relationships; zero payload-duplicate pairs after excluding only ID/source. Retain labels separately; source-related pairs are not canonical product merges.
- **NatMed:** 5 repeated passage-text groups, 13 claim-text groups and 12 claim-span/type groups (these categories overlap). Zero duplicate claim payloads after excluding ID. Keep all 1,269 pending claims with their contexts.
- **Reddit:** 24 repeated record-text groups and 25 passage-text groups. Zero duplicate record payloads after excluding identity fields. Keep thread, parent, timestamp and URL context.
- **Text reuse:** 50 exact-text blobs and 557 source bindings form an optional reuse index. Source records/passages/claims are unchanged. This is not 557 independent evidence items.
- **Terminology:** continue source-namespace/native-ID reuse from preparation; no cross-vocabulary or ingredient/product merge follows from a shared concept link.
- **Outcome:** zero entity merges. Raw files, previous prepared batches and the pilot graph remain unchanged. Product equivalence and prior Medium/Low/no-target mappings remain unresolved where unsupported.

**Next: Step 5 — build the expanded graph** using source-scoped identities, eligible relation-specific mappings, explicit source relationships, provenance and separate review queues. Apply pending clinical and Reddit evidence-role restrictions during export; validate and freeze in Step 6.


## Step 5 — Build the expanded graph

**Collection-wide attributed graph export.** Use the prepared batches, Step 3 mapping decisions and Step 4 identity decisions. Keep source-scoped IDs; no canonical product or ingredient merges.

- Export DSLD labels, ingredient occurrences, servings and source assertions with original values, transformations and quantity-review restrictions. Preserve direct label relationships as source annotations.
- Keep PubChem, RxNorm and MeSH identities separate. Export only High-confidence ingredient mappings with the reviewed relation type; retain typed MeSH literals and URI relationships.
- Export NatMed monographs, subjects, all technically validated **pending** claims, source excerpts, field-support anchors, passages and captured bibliographies. Citation presence does not establish independent study support or product applicability.
- Export Reddit posts, comments, passages and available parent/thread links. Distinguish literal mentions from inherited thread context; no health claims are inferred.
- Keep unavailable references, Medium/Low/no-target mapping queues, source quality issues and duplicate review decisions explicit.

Implementation: `scripts/export_full_graph.py`; policy: `schemas/full_graph_export_v1.json`. Separate node-type CSVs, relationship CSV, provenance JSONL and review files are written to an immutable `data/graph/full-graph-v1-*` directory. Earlier graphs remain unchanged.


In [13]:
from pathlib import Path
import sys, json, csv
full_graph_root = Path.cwd()
if not (full_graph_root / 'scripts').is_dir():
    full_graph_root = full_graph_root.parent
if str(full_graph_root / 'scripts') not in sys.path:
    sys.path.insert(0, str(full_graph_root / 'scripts'))
from export_full_graph import run
full_graph_output, full_graph_report = run(full_graph_root)


data/graph/full-graph-v1-2b5f825bd35bf70d7510
{
  "stage": "Step 5 expanded export; Step 6 representative-question validation and freeze pending",
  "counts": {
    "nodes": 31558,
    "edges": 68292,
    "mapping_edges": 3039,
    "terminology_mapping_edges": 2684,
    "natmed_subject_mapping_edges": 353,
    "additional_category_edges": 2,
    "pending_clinical_claims": 1269,
    "pending_or_excluded_mappings": 4214
  },
  "node_types": {
    "ProductLabel": 400,
    "IngredientOccurrence": 5438,
    "Serving": 400,
    "LabelAssertion": 7514,
    "ExternalRecordReference": 1277,
    "ChemicalCompound": 239,
    "TerminologyConcept": 2647,
    "Monograph": 10,
    "MonographSubject": 10,
    "SourcePassage": 6250,
    "RedditComment": 4593,
    "RedditPost": 287,
    "ClinicalClaim": 1269,
    "SourceExcerpt": 1204,
    "ReferenceDocument": 10,
    "SupplementTopic": 10
  },
  "edge_types": {
    "DECLARES_OCCURRENCE": 5438,
    "HAS_DECLARED_FORM": 1371,
    "HAS_DECLARED_COMPONENT"

In [14]:
print('Node types')
for kind, count in sorted(full_graph_report['node_types'].items()):
    print(f'{kind:28} {count:>7,}')
print('\nMapping and context relationships')
for kind in ['DENOTES_CONCEPT', 'EXACT_MATERIAL_MATCH', 'MEMBER_OF_CATEGORY', 'RELATED_INGREDIENT_SUBJECT', 'RELATED_BOTANICAL_SUBJECT', 'HAS_PENDING_CLAIM', 'MENTIONS_TOPIC', 'INHERITED_THREAD_TOPIC', 'MESH_RELATION']:
    print(f'{kind:28} {full_graph_report["edge_types"].get(kind, 0):>7,}')
print('\nReview and provenance folders:', full_graph_output / 'review', full_graph_output / 'provenance', sep='\n')


Node types
ChemicalCompound                 239
ClinicalClaim                  1,269
ExternalRecordReference        1,277
IngredientOccurrence           5,438
LabelAssertion                 7,514
Monograph                         10
MonographSubject                  10
ProductLabel                     400
RedditComment                  4,593
RedditPost                       287
ReferenceDocument                 10
Serving                          400
SourceExcerpt                  1,204
SourcePassage                  6,250
SupplementTopic                   10
TerminologyConcept             2,647

Mapping and context relationships
DENOTES_CONCEPT                2,677
EXACT_MATERIAL_MATCH               7
MEMBER_OF_CATEGORY                 2
RELATED_INGREDIENT_SUBJECT       351
RELATED_BOTANICAL_SUBJECT          2
HAS_PENDING_CLAIM              1,269
MENTIONS_TOPIC                 1,672
INHERITED_THREAD_TOPIC         6,310
MESH_RELATION                 10,892

Review and provenance folder

In [15]:
# Inspect an exported pending claim and the quantity-review outcomes.
csv.field_size_limit(10_000_000)
with (full_graph_output / 'nodes/ClinicalClaim.csv').open() as handle:
    claim_row = next(csv.DictReader(handle))
claim_data = json.loads(claim_row['properties_json'])
print(json.dumps(dict(claim_id=claim_row['id'], review_state=claim_data['review_state'], clinical_approval=claim_data['clinical_approval'], retrieval_policy=claim_data['retrieval_policy'], original_population=claim_data['original_fields']['population'], original_dose=claim_data['original_fields']['dose'], source_span=claim_data['source_span']), indent=2))
from collections import Counter
quantity_rows = [json.loads(line) for line in (full_graph_output / 'review/dsld_quantity.jsonl').read_text().splitlines()]
print('Quantity comparison gates:', dict(Counter(r['status'] for r in quantity_rows)))
print('These gates remain attached to label assertions; no recommended dose is generated.')


{
  "claim_id": "claim:0008546782b7d059155b42c7a15629bafa6b5311d09a56ad8336ead82c210447",
  "review_state": "pending",
  "clinical_approval": false,
  "retrieval_policy": "attributed_pending_source_excerpt_only",
  "original_population": null,
  "original_dose": {
    "duration": null,
    "frequency": null,
    "maximum": null,
    "minimum": null,
    "qualifier": null,
    "regimen": null,
    "route": null,
    "text": null,
    "unit": null,
    "value": null
  },
  "source_span": {
    "coordinate_system": "unicode_codepoints",
    "end": 26141,
    "file": "data/raw/natmedpro/records.jsonl",
    "pointer": "/1/sections/5/text",
    "sha256": "b413d5ff9051e5ffdbd5c44b804aaf6b9a17f4cdf449e7f60937a7e901b4560c",
    "start": 25562
  }
}
Quantity comparison gates: {'blocked': 905, 'eligible_for_scoped_comparison': 1412, 'pending_review': 75}
These gates remain attached to label assertions; no recommended dose is generated.


### Step 5 checkpoint

- **Export:** `data/graph/full-graph-v1-2b5f825bd35bf70d7510` — **31,558 nodes** and **68,292 relationships**.
- Includes 400 DSLD labels, 5,438 ingredient occurrences, 10 NatMed monographs, 1,269 pending claims, 287 Reddit posts, 4,593 comments and 6,250 source passages, plus terminology and source-reference nodes.
- Activates 2,684 High-confidence terminology links, 353 High-confidence subject links and 2 reviewed category links. Other mappings remain in review files.
- Export checks pass for unique IDs, endpoints, evidence references, mapping eligibility/types and pending clinical status. Zero canonical merges or clinical approvals.
- Duplicate source records remain distinct. The Step 4 exact-text index is retained under `review/identity/`; shared excerpts retain separate claim bindings.

**Next: Step 6 — validate representative POC questions, assess coverage and unresolved issues, then freeze a version.** This export is not yet a frozen POC release. Embeddings, database import and a general answering layer remain later work.


## Step 6 — Validate and freeze a POC version

**Freeze a reproducible data/graph baseline with documented limitations.** This is not clinical approval, a production release or an end-to-end precision/recall evaluation.

- Independently read saved CSVs and verify graph/input hashes, IDs, endpoints, evidence references and exact High-confidence mapping eligibility.
- Resolve original claim excerpts and field-support spans, plus every exported evidence pointer.
- Inspect source-separated retrieval paths for all ten topics, a quantity/formulation comparison, pending Creatine safety retrieval and abstention on unsupported product comparison.
- Document unresolved mapping, quantity, source-quality, clinical-review and navigation gaps.
- Write a separate immutable release manifest pinning the existing graph and each file hash. Future data additions create a new version.

Implementation: `scripts/validate_freeze_full_graph.py`; criteria: `schemas/poc_release_validation_v1.json`.


In [19]:
from pathlib import Path
import sys, json
release_root = Path.cwd()
if not (release_root / 'scripts').is_dir():
    release_root = release_root.parent
if str(release_root / 'scripts') not in sys.path:
    sys.path.insert(0, str(release_root / 'scripts'))
from validate_freeze_full_graph import run
release_output, release_report = run(release_root)


data/graph/poc-release-v1-a9b910bf99750191e874
{
  "status": "passed_with_documented_limitations",
  "checks": {
    "unique_ids": true,
    "endpoints_resolve": true,
    "evidence_resolves": true,
    "high_mapping_eligibility_exact": true,
    "mapping_relations_preserved": true,
    "pending_clinical_status_preserved": true,
    "no_product_clinical_inference": true,
    "excerpts_verified": 1269,
    "field_anchors_verified": 2923,
    "evidence_pointers_verified": 59838
  },
  "cases": 13
}


In [20]:
print(f'{"Topic":15} {"Labels":>7} {"Pending claims":>15} {"Direct Reddit":>14} {"Context Reddit":>15}')
for item in release_report['coverage']:
    print(f'{item["topic"]:15} {item["labels"]:7} {item["pending_claims"]:15} {item["reddit_direct_records"]:14} {item["reddit_context_only_records"]:15}')
    for gap in item['gaps']:
        print('  GAP:', gap)
print('\nLabels can occur under multiple topics; these are graph-path counts, not acquisition totals.')
print('Probiotics has no label path under the current exact ingredientGroup topic rule. This does not establish that raw probiotic labels are absent; topic navigation needs review.')


Topic            Labels  Pending claims  Direct Reddit  Context Reddit
Ashwagandha          40              73            148             519
Creatine             39              76            198             609
Iron                 46              89            150             502
Magnesium            87             168            384            1319
Melatonin            40             159            135             624
Omega-3              32             177            147             787
Probiotics            0              92             91             486
  GAP: No labels on these explicit topic paths
Turmeric             38             146            109             438
Vitamin B12          52              96            143             429
Vitamin D            51             193            167             597

Labels can occur under multiple topics; these are graph-path counts, not acquisition totals.
Probiotics has no label path under the current exact ingredientGroup topic rul

In [21]:
release_cases = json.loads((release_output / 'representative_questions.json').read_text())
for case in release_cases:
    if case['id'].startswith('topic_'):
        continue
    print(case['question'])
    print('Status:', case['status'])
    print(case.get('outcome', case.get('reason', case.get('scope'))))
    print()
print('Unresolved queue rows (overlapping categories):')
print(json.dumps(release_report['unresolved'], indent=2))
print('Frozen release:', release_output)


Do the two BPI Creatine Alkaline labels declare equivalent creatine amounts and identical formulations?
Status: answered_with_limits
Equivalent declared mass and serving basis; other ingredients differ. No product merge or dose recommendation.

What saved Creatine safety statements can be retrieved?
Status: pending_source_retrieval_only
All saved Creatine annotations of type safety, not a clinical safety determination or proof of complete source coverage.

Which BPI product is more effective for athletic performance?
Status: insufficient_evidence
The represented graph has no product-comparative effectiveness assertions. General ingredient claims, label quantities and Reddit discussions cannot establish a product recommendation.

Unresolved queue rows (overlapping categories):
{
  "queue_rows": {
    "dsld_issues": 2845,
    "no_candidate_queries": 4207,
    "mesh_preparation": 745,
    "mapping_pending_excluded": 4214,
    "natmed_no_subject_bridge": 1823,
    "reddit_preparation": 212

### Step 6 checkpoint

- **Frozen graph baseline:** 31,558 nodes, 68,292 relationships, with no source/canonical merges or clinical approvals.
- **13 representative checks:** ten topic-path queries, declared quantity/formulation comparison, pending safety retrieval and unsupported product-comparison abstention. These are fixed graph checks, not a general QA system or clinical benchmark.
- **Source support verified:** 1,269 claim excerpts, 2,923 field-support anchors and 59,838 evidence pointers.
- **Known navigation gap:** Probiotics has NatMed/Reddit coverage but zero DSLD label paths under the exact ingredientGroup-to-topic rule. Preserve this as an open navigation issue; do not force material-identity matches to close it.
- **Other remaining limits:** 4,214 pending/excluded terminology-or-subject mapping rows, 4,207 no-candidate lookup rows and 1,823 ingredient occurrences without a saved NatMed subject bridge. These categories overlap. All 1,269 clinical annotations remain pending; source completeness, quantity conflicts and Reddit context/reference issues remain explicit.
- The separate release manifest pins the existing graph files without modifying or copying the graph. Re-running the validation script verifies the inputs and reproduces the release.

**Preparation/construction stages 1–6 are complete for this limited baseline.** Next review the Probiotics navigation gap, define the downstream evaluation task and establish a retrieval baseline before selecting embeddings or GCN/R-GCN training. A freeze is reproducibility, not a claim that all data issues are resolved.


## Navigation revision — explicit source mentions across all ten topics

The original frozen baseline remains unchanged. Its exact ingredientGroup rule missed Probiotics because DSLD groups are named `Blend`, `Header`, or specific organisms. This separate revision uses a bounded, versioned alias list for all ten topics.

- **DSLD:** add `LABEL_TITLE_MENTIONS_TOPIC` and `INGREDIENT_NAME_MENTIONS_TOPIC`, with original field text, literal match offsets, occurrence role and evidence. Retain existing ingredientGroup links. Count unique labels across channels, not summed edges.
- **NatMed:** preserve source-monograph topics and all pending clinical annotations.
- **Reddit:** preserve direct mentions separately from inherited thread context and retain original hit provenance.
- **PubChem/RxNorm/MeSH:** preserve reviewed identity/concept/category relationships. Topic words do not create identity matches.
- **High confidence means literal text navigation only.** A name such as “iron-free” still mentions iron, but does not assert iron content. Source text and qualifiers remain visible. Species-only probiotic classification, chemical synonyms and clinical applicability are not inferred.

Policy: `schemas/topic_navigation_v1.json`. Implementation: `scripts/expand_topic_navigation.py`. Aliases are bounded explicit spellings, not an exhaustive synonym dictionary; unmatched names remain unlinked under this rule.


In [22]:
from pathlib import Path
import sys, json
navigation_root = Path.cwd()
if not (navigation_root / 'scripts').is_dir():
    navigation_root = navigation_root.parent
if str(navigation_root / 'scripts') not in sys.path:
    sys.path.insert(0, str(navigation_root / 'scripts'))
from expand_topic_navigation import run
navigation_output, navigation_release, navigation_report = run(navigation_root)


data/graph/full-graph-navigation-v1-073a086cbc98af6632ae
data/graph/poc-navigation-release-v1-29919793c9d8bc4951d0
{
  "added": 1356,
  "sources": {
    "dsld": {
      "added_title_mentions": 396,
      "added_ingredient_mentions": 960
    },
    "natmed": {
      "source_topic_links": 10,
      "pending_claim_links": 1269,
      "changed": 0
    },
    "reddit": {
      "direct_mentions": 1672,
      "inherited_context": 6310,
      "changed": 0
    },
    "terminology": {
      "by_source": {
        "mesh": 1230,
        "rxnorm": 1452,
        "pubchem": 4
      },
      "changed": 0
    }
  }
}


In [23]:
print(f'{"Topic":15} {"Group labels":>12} {"Name labels":>12} {"Title labels":>12} {"Unique total":>13} {"New labels":>11}')
for row in navigation_report['after']:
    print(f'{row["topic"]:15} {row["group_labels"]:12} {row["ingredient_name_labels"]:12} {row["title_labels"]:12} {row["all_navigation_labels"]:13} {row["newly_reachable_labels"]:11}')
print('\nChannels overlap; a topic mention does not establish composition or clinical applicability.')
print('\nCross-source preservation checks:')
print(json.dumps(navigation_report['source_audit'], indent=2))
print('\nCurrent navigation graph:', navigation_output)
print('Frozen navigation release:', navigation_release)


Topic           Group labels  Name labels Title labels  Unique total  New labels
Ashwagandha               40           39           40            40           0
Creatine                  39           40           39            40           1
Iron                      46           46           40            46           0
Magnesium                 87          188           45           188         101
Melatonin                 40           40           40            40           0
Omega-3                   32           36           40            40           8
Probiotics                 0           38           40            40          40
Turmeric                  38           43           40            43           5
Vitamin B12               52           56           32            56           4
Vitamin D                 51           52           40            52           1

Channels overlap; a topic mention does not establish composition or clinical applicability.

Cross-source pr

### Navigation revision outcome

- Added **1,356 literal navigation links**: 396 product-title links and 960 ingredient-name links across all ten supplement topics. Different evidence channels remain distinct; label result counts deduplicate by label ID.
- **Probiotics:** 38 labels have explicit ingredient-name mentions; 40 have explicit product-title mentions. Their union is **40 labels**, closing the original topic-navigation gap without species/strain inference.
- New graph: **31,558 nodes and 69,648 relationships**. Existing nodes, 68,292 relationships, clinical annotations, mapping confidence and review queues remain unchanged.
- NatMed retains 10 source-topic links and 1,269 pending claims. Reddit retains 1,672 direct mentions and 6,310 inherited-context links. All terminology mappings remain unchanged.
- Separate graph and release manifests preserve the original frozen baseline and pin this navigation revision. Earlier sections report the historical baseline; use the revision output printed above for current topic navigation.
- Remaining limitations are unchanged: pending mappings and clinical review, incomplete source coverage, quantity conflicts, and missing Reddit references. Literal-match navigation is not a clinical or material-identity guarantee.

**Next:** define representative retrieval tasks and evaluation examples before deciding whether embeddings or a GNN improve the baseline.


### Statement-preservation revision — collection-wide fix

**Current graph:** `data/graph/full-graph-statements-v1-27c544f4940e7162944c`  
**Frozen revision:** `data/graph/poc-statements-release-v1-684ce1012b4f76e0af1a`

All **6,462 source statements across 400 captured DSLD labels** are now preserved and connected to their product labels. The previous graph retained 2,022 statements-array entries; this revision adds 4,440 omitted entries. (The older 2,023 label-statement count also included one other-ingredient text assertion.)

The rule is **preserve all captured statement types**, not “classify every statement as a warning.” Original types and wording remain unchanged, including marketing, disclaimers, allergen information and general-content restrictions. Warning relevance remains a separate content-review decision. No source/clinical corrections, identity merges or mapping promotions were made.

The revision has **35,998 nodes and 74,088 relationships**. Existing nodes, relationships, sources and frozen versions remain intact. Full captured-statement coverage is not a claim of label-image completeness or clinical correctness.

Reproduce with `scripts/preserve_dsld_statements.py`; policy: `schemas/dsld_statement_preservation_v1.json`.


In [24]:
from pathlib import Path
import json, hashlib
statement_root = Path.cwd()
if not (statement_root / 'data').is_dir():
    statement_root = statement_root.parent
statement_graph = statement_root / 'data/graph/full-graph-statements-v1-27c544f4940e7162944c'
statement_release = statement_root / 'data/graph/poc-statements-release-v1-684ce1012b4f76e0af1a'
statement_freeze = json.loads((statement_release / 'freeze_manifest.json').read_text())
assert hashlib.sha256((statement_graph / 'manifest.json').read_bytes()).hexdigest() == statement_freeze['graph_manifest_sha256']
statement_report = json.loads((statement_graph / 'validation_report.json').read_text())
print(json.dumps({k: statement_report[k] for k in ['labels','source_statement_occurrences','previously_preserved','added_statements','nodes','edges','checks']}, indent=2))


{
  "labels": 400,
  "source_statement_occurrences": 6462,
  "previously_preserved": 2022,
  "added_statements": 4440,
  "nodes": 35998,
  "edges": 74088,
  "checks": {
    "all_captured_statements_preserved": true,
    "all_statement_paths_reachable": true,
    "existing_assertions_and_edges_unchanged": true,
    "no_clinical_approval": true,
    "no_mapping_changes": true,
    "source_values_and_hashes_verified": true
  }
}
